# Diagnóstico de calidad de datos – Proyecto League of Legends

**Aprendiz:** Juan Camilo Montes Velásquez
**Programa:** Tecnólogo en Análisis y Desarrollo de Software, SENA
**Evidencia:** GA1-220501094-01-AA1-EV05 – Diagnosticar la calidad de datos: preparación, exploración y documentación

*Nota: los dos CSV deben estar en `data/raw/` (ver README para descargarlos).*

## Momento 1. Reconocimiento del proyecto y de los datasets

### Nombre y propósito del proyecto
**¿Qué gana una partida de League of Legends?** El propósito es identificar qué variables de los primeros minutos de una partida pesan más en el resultado, para que jugadores y entrenadores sepan en qué enfocarse al inicio basándose en datos y no en opiniones.

### Pregunta analítica
En la EV02 planteé: *¿Qué variables de los primeros 15 minutos (diferencia de oro, torres y objetivos neutrales) influyen más en la probabilidad de ganar en partidas ranked, y con qué precisión se puede predecir el resultado?*

Al revisar las fuentes en la EV03 vi que el dataset de Kaggle solo llega al minuto 10. Por eso ajusté la pregunta para que los datos sí puedan responderla:

> **¿Qué variables de los primeros 15 minutos de una partida (diferencia de oro, torres y objetivos neutrales) influyen más en la probabilidad de ganar en partidas profesionales de League of Legends, y se mantienen en partidas ranked de alto nivel?**

### Datasets seleccionados

| Dataset | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|
| `2025_LoL_esports_match_data_from_OraclesElixir.csv` | Oracle's Elixir (identificada en la EV02) | Partidas profesionales de 2025 de muchas ligas. Por cada partida hay una fila por jugador y una por equipo, con oro, experiencia y objetivos a los 10, 15, 20 y 25 minutos, y el resultado. | Dataset principal: tiene las variables a los 15 minutos y el resultado (`result`). |
| `high_diamond_ranked_10min.csv` | Kaggle, usuario bobbyscience (seleccionado en la EV03) | Unas 10.000 partidas ranked de nivel Diamante a Maestro, con estadísticas de ambos equipos en los primeros 10 minutos y si ganó el equipo azul. | Sirve para comparar si las mismas variables pesan igual en partidas ranked. |

La API de Riot (la tercera fuente de la EV03) no la uso por ahora porque exige una clave de desarrollador y tiene límites de consultas.

### Aspectos de calidad que debo revisar
- **Valores faltantes:** sobre todo en las variables a los 15 minutos, que son las de mi pregunta.
- **Tipos de datos:** que las fechas, los parches y los identificadores tengan un tipo coherente.
- **Estructura de las filas:** qué representa cada fila (jugador, equipo o partida), para no contar mal.
- **Inconsistencias:** valores imposibles o contradictorios (por ejemplo, dos equipos con el primer dragón en la misma partida).
- **Cobertura:** qué ligas, minutos y niveles de juego cubre cada dataset.

## Momento 2. Preparación del entorno y carga de datos

In [1]:
import warnings
import pandas as pd

# En pandas 3 el texto tiene su propio tipo (str). Al pedir describe(include="object")
# pandas avisa que eso cambiará en el futuro; oculto solo ese aviso para que la salida se lea mejor.
warnings.filterwarnings("ignore", message=".*'str' dtypes are included.*")
pd.set_option("display.max_columns", 30)

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.1.1


In [2]:
RUTA = "../data/raw/"

# low_memory=False hace que pandas lea todo el archivo antes de decidir el tipo de cada columna
# (sin esto avisa de columnas con tipos mezclados, porque el archivo es grande).
partidas_pro = pd.read_csv(RUTA + "2025_LoL_esports_match_data_from_OraclesElixir.csv", low_memory=False)
partidas_ranked = pd.read_csv(RUTA + "high_diamond_ranked_10min.csv")

dfs = {
    "partidas_pro (Oracle's Elixir)": partidas_pro,
    "partidas_ranked (Kaggle)": partidas_ranked,
}

for nombre, df in dfs.items():
    print(f"{nombre}: cargado con {df.shape[0]} filas y {df.shape[1]} columnas")

partidas_pro (Oracle's Elixir): cargado con 120528 filas y 165 columnas
partidas_ranked (Kaggle): cargado con 9879 filas y 40 columnas


Los dos archivos se cargaron sin errores. Ambos usan coma como separador, así que no hace falta `sep`.

## Momento 3. Inspección de la estructura de los datasets

### Dataset 1: partidas profesionales (Oracle's Elixir)

In [3]:
print("Primeros 5 registros:")
display(partidas_pro.head())
print("Últimos 5 registros:")
display(partidas_pro.tail())

Primeros 5 registros:


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,participantid,side,position,playername,playerid,...,goldat25,xpat25,csat25,opp_goldat25,opp_xpat25,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,1,Blue,top,PatkicaA,oe:player:c659697694306de62d978569b84c344,...,9244.0,12552.0,217.0,9020.0,12553.0,200.0,224.0,-1.0,17.0,1.0,1.0,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,2,Blue,jng,Joinze,oe:player:dbdc61a1c41acedcbc7d399727155ac,...,7040.0,8877.0,139.0,9403.0,10321.0,157.0,-2363.0,-1444.0,-18.0,0.0,1.0,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,3,Blue,mid,Sayn,oe:player:694d028e62f4ea668b206ab752b6f94,...,8488.0,10669.0,200.0,10040.0,13134.0,241.0,-1552.0,-2465.0,-41.0,1.0,0.0,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,4,Blue,bot,Shiganari,oe:player:90704735ca9fc01f2244f23f6e5d635,...,9097.0,11763.0,251.0,11710.0,12919.0,257.0,-2613.0,-1156.0,-6.0,1.0,1.0,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,5,Blue,sup,Lekcyc,oe:player:74f3f60a44ee916ecc257a5381be756,...,5357.0,6259.0,38.0,6019.0,6993.0,20.0,-662.0,-734.0,18.0,0.0,2.0,2.0,0.0,8.0,0.0


Últimos 5 registros:


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,participantid,side,position,playername,playerid,...,goldat25,xpat25,csat25,opp_goldat25,opp_xpat25,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
120523,LOLTMNT03_332179,complete,NaN,DCup,2025,Season,0,2025-12-29 10:52:27,3,15.24,8,Red,mid,Tangyuan,oe:player:3c93fcbfb525315f6412904e5b9abbd,...,11409.0,13632.0,192.0,11534.0,14132.0,224.0,-125.0,-500.0,-32.0,8.0,5.0,2.0,4.0,4.0,1.0
120524,LOLTMNT03_332179,complete,NaN,DCup,2025,Season,0,2025-12-29 10:52:27,3,15.24,9,Red,bot,Shaoye,oe:player:6d734e1e86b978852d95cfd63d6a643,...,9971.0,11356.0,226.0,10512.0,10843.0,221.0,-541.0,513.0,5.0,0.0,7.0,2.0,2.0,4.0,2.0
120525,LOLTMNT03_332179,complete,NaN,DCup,2025,Season,0,2025-12-29 10:52:27,3,15.24,10,Red,sup,Ycx,oe:player:397bcf46311de3cf780469b2a842b67,...,6773.0,8323.0,48.0,5518.0,5924.0,31.0,1255.0,2399.0,17.0,1.0,9.0,1.0,1.0,1.0,4.0
120526,LOLTMNT03_332179,complete,NaN,DCup,2025,Season,0,2025-12-29 10:52:27,3,15.24,100,Blue,team,NaN,NaN,...,43636.0,51932.0,808.0,47738.0,60226.0,874.0,-4102.0,-8294.0,-66.0,8.0,18.0,13.0,13.0,34.0,8.0
120527,LOLTMNT03_332179,complete,NaN,DCup,2025,Season,0,2025-12-29 10:52:27,3,15.24,200,Red,team,NaN,NaN,...,47738.0,60226.0,874.0,43636.0,51932.0,808.0,4102.0,8294.0,66.0,13.0,34.0,8.0,8.0,18.0,13.0


In [4]:
print("shape:", partidas_pro.shape)
print()
print("columns:")
print(list(partidas_pro.columns))

shape: (120528, 165)

columns:
['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp

In [5]:
print(partidas_pro.dtypes.value_counts())
print()
print(partidas_pro.dtypes.to_string())

float64    122
str         23
int64       20
Name: count, dtype: int64

gameid                          str
datacompleteness                str
url                             str
league                          str
year                          int64
split                           str
playoffs                      int64
date                            str
game                          int64
patch                       float64
participantid                 int64
side                            str
position                        str
playername                      str
playerid                        str
teamname                        str
teamid                          str
firstPick                   float64
champion                        str
ban1                            str
ban2                            str
ban3                            str
ban4                            str
ban5                            str
pick1                           str
pick2                       

In [6]:
partidas_pro.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 120528 entries, 0 to 120527
Columns: 165 entries, gameid to opp_deathsat25
dtypes: float64(122), int64(20), str(23)
memory usage: 151.7 MB


In [7]:
partidas_pro.describe()

,year,playoffs,game,patch,participantid,firstPick,gamelength,result,kills,deaths,assists,teamkills,teamdeaths,doublekills,triplekills,...,goldat25,xpat25,csat25,opp_goldat25,opp_xpat25,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
count,120528.000000,120528.000000,120528.000000,120528.000000,120528.000000,120384.000000,120528.000000,120528.000000,120528.000000,120528.000000,120528.000000,120528.000000,120528.000000,110688.000000,110688.000000,...,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000,106596.00000,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000,106596.000000
mean,2025.024691,0.216448,1.832139,15.103831,29.583333,0.500000,1921.665571,0.500000,5.432480,5.445042,12.524525,16.297441,16.335125,0.625741,0.111250,...,14695.741210,18289.702297,283.673459,14695.741210,18289.702297,283.673459,0.00000,0.000000,0.000000,3.433018,7.262468,3.441949,3.433018,7.262468,3.441949
std,0.155183,0.411825,0.983287,0.054894,57.650707,0.500002,327.769991,0.500002,6.487047,6.210628,14.657652,8.120731,8.120701,1.112181,0.367808,...,13321.916607,16617.299912,264.330705,13321.916607,16617.299912,264.330705,2847.02085,3317.191838,34.951771,4.286482,8.859333,4.127757,4.286482,8.859333,4.127757
min,2025.000000,0.000000,1.000000,15.010000,1.000000,0.000000,1059.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4722.000000,3383.000000,6.000000,4722.000000,3383.000000,6.000000,-19544.00000,-26182.000000,-274.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025.000000,0.000000,1.000000,15.070000,3.750000,0.000000,1696.000000,0.000000,1.000000,2.000000,4.000000,10.000000,10.000000,0.000000,0.000000,...,8191.000000,10266.000000,165.000000,8191.000000,10266.000000,165.000000,-1173.00000,-1392.000000,-17.000000,1.000000,2.000000,1.000000,1.000000,2.000000,1.000000
50%,2025.000000,0.000000,2.000000,15.100000,6.500000,0.500000,1856.000000,0.500000,3.000000,4.000000,8.000000,16.000000,16.000000,0.000000,0.000000,...,9398.000000,12023.000000,208.000000,9398.000000,12023.000000,208.000000,0.00000,0.000000,0.000000,2.000000,4.000000,2.000000,2.000000,4.000000,2.000000
75%,2025.000000,0.000000,2.000000,15.150000,9.250000,1.000000,2103.000000,1.000000,7.000000,6.000000,14.000000,22.000000,22.000000,1.000000,0.000000,...,10923.000000,13821.000000,243.000000,10923.000000,13821.000000,243.000000,1173.00000,1392.000000,17.000000,4.000000,8.000000,4.000000,4.000000,8.000000,4.000000
max,2026.000000,1.000000,5.000000,15.240000,200.000000,1.000000,3621.000000,1.000000,48.000000,48.000000,128.000000,48.000000,48.000000,10.000000,4.000000,...,60330.000000,73801.000000,1066.000000,60330.000000,73801.000000,1066.000000,19544.00000,26182.000000,274.000000,42.000000,90.000000,42.000000,42.000000,90.000000,42.000000


In [8]:
partidas_pro.describe(include="object")

,gameid,datacompleteness,url,league,split,date,side,position,playername,playerid,teamname,teamid,champion,ban1,ban2,ban3,ban4,ban5,pick1,pick2,pick3,pick4,pick5
count,120528,120528,9840,120528,104184,120528,120528,120528,100440,98854,120528,116892,100440,120114,120234,120150,119796,119784,20036,20036,20036,20036,20036
unique,10044,2,321,45,20,10036,2,6,2803,2636,444,382,172,156,153,157,164,166,140,159,165,168,171
top,LOLTMNT03_179647,complete,https://lpl.qq.com/es/stats.shtml?bmid=11738,LPL,Spring,2025-02-05 20:08:47,Blue,top,Shanks,oe:player:41deab55354f57a94135989133f9968,Bilibili Gaming,oe:team:d356a144644879dabb5f34cd99c886d,Rell,Pantheon,Varus,Varus,Braum,Alistar,Varus,Xin Zhao,Ambessa,Rell,Rell
freq,12,110688,60,9660,19056,24,60264,20088,238,200,1134,1134,3017,7206,6846,7140,5058,5070,1431,1244,778,905,1009


**Características principales:** son 120.528 filas y 165 columnas. Hay columnas de texto (liga, equipo, jugador, campeón, fecha), enteros (resultado, kills) y muchas decimales (estadísticas a los 10, 15, 20 y 25 minutos). Con `head()` se ve que las filas de una misma partida se repiten con el mismo `gameid`: una fila no es una partida. Lo compruebo abajo.

In [9]:
print("Filas:", len(partidas_pro))
print("Partidas distintas (gameid):", partidas_pro["gameid"].nunique())
print("Filas por partida:", partidas_pro.groupby("gameid").size().unique())
print()
print("Valores de position:")
print(partidas_pro["position"].value_counts())

Filas: 120528
Partidas distintas (gameid): 10044
Filas por partida: [12]

Valores de position:
position
top     20088
jng     20088
mid     20088
bot     20088
sup     20088
team    20088
Name: count, dtype: int64


Cada partida tiene **12 filas**: 10 de jugadores (`top`, `jng`, `mid`, `bot`, `sup` de cada equipo) y 2 de equipo (`position == "team"`). Hay 10.044 partidas, no 120.528. Para mi pregunta (qué hace ganar a un equipo) las filas que sirven son las de equipo.

### Dataset 2: partidas ranked Diamante (Kaggle)

In [10]:
print("Primeros 5 registros:")
display(partidas_ranked.head())
print("Últimos 5 registros:")
display(partidas_ranked.tail())

Primeros 5 registros:


,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,...,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,0,0,17210,6.6,17039,...,9,8,0,0,0,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,0,0,14712,6.6,16265,...,5,2,2,1,1,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,0,0,16113,6.4,16221,...,7,14,0,0,0,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,1,0,15157,7.0,17954,...,4,10,0,0,0,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,0,0,16400,7.0,18543,...,6,7,1,1,0,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4


Últimos 5 registros:


,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,...,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
9874,4527873286,1,17,2,1,7,4,5,1,1,0,0,17765,7.2,18967,...,7,7,0,0,0,0,15246,6.8,16498,229,34,-2519,-2469,22.9,1524.6
9875,4527797466,1,54,0,0,6,4,8,1,1,0,0,16238,7.2,19255,...,6,3,0,0,0,0,15456,7.0,18367,206,56,-782,-888,20.6,1545.6
9876,4527713716,0,23,1,0,6,7,5,0,0,0,0,15903,7.0,18032,...,6,11,1,1,0,0,18319,7.4,19909,261,60,2416,1877,26.1,1831.9
9877,4527628313,0,14,4,1,2,3,3,1,1,0,0,14459,6.6,17229,...,2,1,0,0,0,0,15298,7.2,18314,247,40,839,1085,24.7,1529.8
9878,4523772935,1,18,0,1,6,6,5,0,0,0,0,16266,7.0,17321,...,6,4,1,1,0,0,15339,6.8,17379,201,46,-927,58,20.1,1533.9


In [11]:
print("shape:", partidas_ranked.shape)
print()
print("columns:")
print(list(partidas_ranked.columns))

shape: (9879, 40)

columns:
['gameId', 'blueWins', 'blueWardsPlaced', 'blueWardsDestroyed', 'blueFirstBlood', 'blueKills', 'blueDeaths', 'blueAssists', 'blueEliteMonsters', 'blueDragons', 'blueHeralds', 'blueTowersDestroyed', 'blueTotalGold', 'blueAvgLevel', 'blueTotalExperience', 'blueTotalMinionsKilled', 'blueTotalJungleMinionsKilled', 'blueGoldDiff', 'blueExperienceDiff', 'blueCSPerMin', 'blueGoldPerMin', 'redWardsPlaced', 'redWardsDestroyed', 'redFirstBlood', 'redKills', 'redDeaths', 'redAssists', 'redEliteMonsters', 'redDragons', 'redHeralds', 'redTowersDestroyed', 'redTotalGold', 'redAvgLevel', 'redTotalExperience', 'redTotalMinionsKilled', 'redTotalJungleMinionsKilled', 'redGoldDiff', 'redExperienceDiff', 'redCSPerMin', 'redGoldPerMin']


In [12]:
print(partidas_ranked.dtypes.value_counts())
print()
print(partidas_ranked.dtypes.to_string())

int64      34
float64     6
Name: count, dtype: int64

gameId                            int64
blueWins                          int64
blueWardsPlaced                   int64
blueWardsDestroyed                int64
blueFirstBlood                    int64
blueKills                         int64
blueDeaths                        int64
blueAssists                       int64
blueEliteMonsters                 int64
blueDragons                       int64
blueHeralds                       int64
blueTowersDestroyed               int64
blueTotalGold                     int64
blueAvgLevel                    float64
blueTotalExperience               int64
blueTotalMinionsKilled            int64
blueTotalJungleMinionsKilled      int64
blueGoldDiff                      int64
blueExperienceDiff                int64
blueCSPerMin                    float64
blueGoldPerMin                  float64
redWardsPlaced                    int64
redWardsDestroyed                 int64
redFirstBlood            

In [13]:
partidas_ranked.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 9879 entries, 0 to 9878
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   gameId                        9879 non-null   int64  
 1   blueWins                      9879 non-null   int64  
 2   blueWardsPlaced               9879 non-null   int64  
 3   blueWardsDestroyed            9879 non-null   int64  
 4   blueFirstBlood                9879 non-null   int64  
 5   blueKills                     9879 non-null   int64  
 6   blueDeaths                    9879 non-null   int64  
 7   blueAssists                   9879 non-null   int64  
 8   blueEliteMonsters             9879 non-null   int64  
 9   blueDragons                   9879 non-null   int64  
 10  blueHeralds                   9879 non-null   int64  
 11  blueTowersDestroyed           9879 non-null   int64  
 12  blueTotalGold                 9879 non-null   int64  
 13  blueAvgLevel  

In [14]:
partidas_ranked.describe()

,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,...,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
count,9.879000e+03,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,...,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000
mean,4.500084e+09,0.499038,22.288288,2.824881,0.504808,6.183925,6.137666,6.645106,0.549954,0.361980,0.187974,0.051422,16503.455512,6.916004,17928.110133,...,6.183925,6.662112,0.573135,0.413098,0.160036,0.043021,16489.041401,6.925316,17961.730438,217.349226,51.313088,-14.414111,33.620306,21.734923,1648.904140
std,2.757328e+07,0.500024,18.019177,2.174998,0.500002,3.011028,2.933818,4.064520,0.625527,0.480597,0.390712,0.244369,1535.446636,0.305146,1200.523764,...,3.011028,4.060612,0.626482,0.492415,0.366658,0.216900,1490.888406,0.305311,1198.583912,21.911668,10.027885,2453.349179,1920.370438,2.191167,149.088841
min,4.295358e+09,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10730.000000,4.600000,10098.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11212.000000,4.800000,10465.000000,107.000000,4.000000,-11467.000000,-8348.000000,10.700000,1121.200000
25%,4.483301e+09,0.000000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,15415.500000,6.800000,17168.000000,...,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,15427.500000,6.800000,17209.500000,203.000000,44.000000,-1596.000000,-1212.000000,20.300000,1542.750000
50%,4.510920e+09,0.000000,16.000000,3.000000,1.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.000000,16398.000000,7.000000,17951.000000,...,6.000000,6.000000,0.000000,0.000000,0.000000,0.000000,16378.000000,7.000000,17974.000000,218.000000,51.000000,-14.000000,28.000000,21.800000,1637.800000
75%,4.521733e+09,1.000000,20.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.000000,17459.000000,7.200000,18724.000000,...,8.000000,9.000000,1.000000,1.000000,0.000000,0.000000,17418.500000,7.200000,18764.500000,233.000000,57.000000,1585.500000,1290.500000,23.300000,1741.850000
max,4.527991e+09,1.000000,250.000000,27.000000,1.000000,22.000000,22.000000,29.000000,2.000000,1.000000,1.000000,4.000000,23701.000000,8.000000,22224.000000,...,22.000000,28.000000,2.000000,1.000000,1.000000,2.000000,22732.000000,8.200000,22269.000000,289.000000,92.000000,10830.000000,9333.000000,28.900000,2273.200000


In [15]:
# describe(include="object") falla si el DataFrame no tiene columnas de texto,
# así que primero reviso si hay alguna.
texto = partidas_ranked.select_dtypes(include="object").columns
print("Columnas de texto:", list(texto))

if len(texto) > 0:
    display(partidas_ranked.describe(include="object"))
else:
    print("Este dataset no tiene columnas de tipo object, por eso describe(include='object') no aplica.")

Columnas de texto: []
Este dataset no tiene columnas de tipo object, por eso describe(include='object') no aplica.


**Características principales:** son 9.879 filas y 40 columnas, todas numéricas. Aquí sí cada fila es una partida (`gameId` no se repite). Las columnas vienen en pares: las que empiezan por `blue` son del equipo azul y las que empiezan por `red` del rojo. La variable a predecir es `blueWins` (1 si ganó el azul).

### Tabla de diagnóstico estructural

In [16]:
def resumen_tipos(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{n} {t}" for t, n in conteo.items())

estructura = pd.DataFrame({
    "dataset": list(dfs.keys()),
    "filas": [df.shape[0] for df in dfs.values()],
    "columnas": [df.shape[1] for df in dfs.values()],
    "tipos_de_datos": [resumen_tipos(df) for df in dfs.values()],
    "observaciones": [
        "12 filas por partida (10 jugadores + 2 equipos): 10.044 partidas. Muchas columnas con nulos. "
        "Mezcla texto, enteros y decimales.",
        "1 fila por partida (9.879 partidas). Sin columnas de texto. Datos solo hasta el minuto 10. "
        "Columnas en pares azul/rojo.",
    ],
})

pd.set_option("display.max_colwidth", None)
estructura

,dataset,filas,columnas,tipos_de_datos,observaciones
0,partidas_pro (Oracle's Elixir),120528,165,"122 float64, 23 str, 20 int64","12 filas por partida (10 jugadores + 2 equipos): 10.044 partidas. Muchas columnas con nulos. Mezcla texto, enteros y decimales."
1,partidas_ranked (Kaggle),9879,40,"34 int64, 6 float64",1 fila por partida (9.879 partidas). Sin columnas de texto. Datos solo hasta el minuto 10. Columnas en pares azul/rojo.


## Momento 4. Diagnóstico inicial de calidad

### 4.1 Valores faltantes

En el dataset profesional, comparar `info()` con el total de filas muestra muchas columnas con nulos. Pero no todos significan lo mismo, porque hay filas de jugador y filas de equipo. Los separo para entenderlos.

In [17]:
nulos = partidas_pro.isna().sum()
print("Columnas con al menos un nulo:", (nulos > 0).sum(), "de", partidas_pro.shape[1])
print()
print("Las 15 columnas con más nulos:")
print(nulos.sort_values(ascending=False).head(15))

Columnas con al menos un nulo: 129 de 165

Las 15 columnas con más nulos:
dragons (type unknown)     118888
monsterkillsownjungle      110688
monsterkillsenemyjungle    110688
url                        110688
atakhans                   102132
opp_atakhans               102132
opp_elementaldrakes        102080
firstherald                102080
infernals                  102080
hextechs                   102080
elders                     102080
opp_elders                 102080
oceans                     102080
elementaldrakes            102080
clouds                     102080
dtype: int64


In [18]:
equipos = partidas_pro[partidas_pro["position"] == "team"]
jugadores = partidas_pro[partidas_pro["position"] != "team"]

solo_nulas_en_equipo = [c for c in partidas_pro.columns if equipos[c].isna().all()]
solo_nulas_en_jugador = [c for c in partidas_pro.columns if jugadores[c].isna().all()]

print("Columnas siempre vacías en las filas de EQUIPO:", solo_nulas_en_equipo)
print()
print("Columnas siempre vacías en las filas de JUGADOR:", solo_nulas_en_jugador)

Columnas siempre vacías en las filas de EQUIPO: ['playername', 'playerid', 'champion', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'damageshare', 'earnedgoldshare', 'total cs']

Columnas siempre vacías en las filas de JUGADOR: ['pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'gspd', 'gpr']


Gran parte de los nulos **no son errores, sino datos que no aplican**: una fila de equipo no tiene `champion` ni `playername`, y una fila de jugador no tiene `towers` ni `firstdragon`, porque esos objetivos son del equipo.

Lo que más me importa es si faltan las variables a los 15 minutos en las filas de equipo:

In [19]:
cols_15 = ["golddiffat15", "xpdiffat15", "csdiffat15", "firstherald", "firstdragon", "firsttower", "firstblood"]

print("Nulos en filas de equipo:")
print(equipos[cols_15].isna().sum())
print()
print("Filas de equipo por datacompleteness:")
print(equipos["datacompleteness"].value_counts())
print()
print("Nulos de golddiffat15 según datacompleteness:")
print(equipos.groupby("datacompleteness")["golddiffat15"].apply(lambda s: s.isna().sum()))

Nulos en filas de equipo:
golddiffat15    1640
xpdiffat15      1640
csdiffat15      1640
firstherald     1640
firstdragon        0
firsttower         0
firstblood         0
dtype: int64

Filas de equipo por datacompleteness:
datacompleteness
complete    18448
partial      1640
Name: count, dtype: int64

Nulos de golddiffat15 según datacompleteness:
datacompleteness
complete       0
partial     1640
Name: golddiffat15, dtype: int64


In [20]:
parciales = equipos[equipos["datacompleteness"] == "partial"]
print("Partidas con datos parciales:", parciales["gameid"].nunique())
print()
print("Ligas de esas partidas:")
print(parciales["league"].value_counts())
print()
print("Partidas de la LPL en total:", equipos.loc[equipos["league"] == "LPL", "gameid"].nunique())

Partidas con datos parciales: 820

Ligas de esas partidas:
league
LPL     1610
WLDs      24
DCup       6
Name: count, dtype: int64

Partidas de la LPL en total: 805


Los nulos de `golddiffat15`, `xpdiffat15`, `csdiffat15` y `firstherald` aparecen **solo en las partidas marcadas como `partial`** (820 partidas). Casi todas son de la **LPL** (la liga china), y **todas** las partidas de la LPL están incompletas. O sea, la liga más grande del dataset no sirve para responder mi pregunta a los 15 minutos.

Otras columnas con nulos:

In [21]:
for col in ["url", "split", "teamid", "playerid", "ban1"]:
    print(f"{col}: {partidas_pro[col].isna().sum()} nulos")

print()
print("Ligas con más teamid vacío:")
print(partidas_pro.loc[partidas_pro["teamid"].isna(), "league"].value_counts().head())

url: 110688 nulos
split: 16344 nulos
teamid: 3636 nulos
playerid: 21674 nulos
ban1: 414 nulos

Ligas con más teamid vacío:
league
LJL      2022
CD        468
LPLOL     330
PCS       150
LRN       126
Name: count, dtype: int64


In [22]:
print("Nulos en el dataset ranked:", partidas_ranked.isna().sum().sum())

Nulos en el dataset ranked: 0


- `url` solo está en las partidas parciales (es el enlace de donde se tomaron), así que no afecta.
- `split` falta en torneos que no se dividen en splits (por ejemplo Worlds o MSI).
- `teamid` y `playerid` faltan en algunas ligas menores. Afectaría si quisiera seguir a un equipo o jugador, pero no para mi pregunta.
- `ban1` falta en pocas filas (bans no registrados).
- El dataset ranked **no tiene ningún valor nulo**.

### 4.2 Tipos de datos

In [23]:
print("patch:", partidas_pro["patch"].dtype, "->", sorted(partidas_pro["patch"].unique())[:12])
print("date:", partidas_pro["date"].dtype, "->", partidas_pro["date"].head(2).tolist())
print("firstdragon:", partidas_pro["firstdragon"].dtype, "->", partidas_pro["firstdragon"].dropna().unique())
print("result:", partidas_pro["result"].dtype, "->", partidas_pro["result"].unique())
print("gameId (ranked):", partidas_ranked["gameId"].dtype, "->", partidas_ranked["gameId"].head(2).tolist())

patch: float64 -> [np.float64(15.01), np.float64(15.02), np.float64(15.03), np.float64(15.04), np.float64(15.05), np.float64(15.06), np.float64(15.07), np.float64(15.08), np.float64(15.09), np.float64(15.1), np.float64(15.11), np.float64(15.12)]
date: str -> ['2025-01-11 11:11:24', '2025-01-11 11:11:24']
firstdragon: float64 -> [0. 1.]
result: int64 -> [0 1]
gameId (ranked): int64 -> [4519157822, 4523371949]


- **`patch` es decimal (`float64`)**, pero un parche es una etiqueta, no un número. Como decimal, el parche 15.10 aparece como `15.1` y el 15.20 como `15.2`, lo que se puede confundir con un parche 15.1 o 15.2. Debería ser texto.
- **`date` es texto (`str`)**, no fecha. Así no se puede ordenar ni filtrar por mes de forma confiable. Debería convertirse con `pd.to_datetime()`.
- **`firstdragon`, `firstherald`, `firsttower` y `firstblood` son decimales**, aunque solo valen 0 o 1. Pandas los guarda así porque tienen nulos (en las filas de jugador).
- **`gameId` (ranked) es entero**, pero es un identificador: no tiene sentido sumarlo ni promediarlo (en `describe()` aparece su "promedio", que no significa nada).
- `result` y `blueWins` son enteros 0/1, que es correcto.

### 4.3 Posibles inconsistencias

In [24]:
# En cada partida, exactamente un equipo debería tener cada "primer" objetivo.
for col in ["firstblood", "firstdragon", "firsttower", "firstherald"]:
    suma = equipos.groupby("gameid")[col].sum(min_count=1)
    print(col, "-> equipos con el objetivo por partida:", suma.value_counts(dropna=False).to_dict())

print()
print("Ganadores por partida:", equipos.groupby("gameid")["result"].sum().value_counts().to_dict())

firstblood -> equipos con el objetivo por partida: {1.0: 10043, 0.0: 1}
firstdragon -> equipos con el objetivo por partida: {1.0: 10037, 0.0: 7}
firsttower -> equipos con el objetivo por partida: {1.0: 10044}
firstherald -> equipos con el objetivo por partida: {1.0: 9205, nan: 820, 0.0: 19}

Ganadores por partida: {1: 10044}


- En todas las partidas hay **exactamente un ganador**, y `firsttower` siempre lo tiene un solo equipo. Eso está bien.
- Hay 1 partida sin `firstblood`, 7 sin `firstdragon` y 19 sin `firstherald` (ningún equipo lo tiene). Puede ser real (nadie tomó ese objetivo antes de terminar) o un dato no registrado; hay que revisarlas.
- Las 820 partidas con `firstherald` nulo son las parciales ya vistas.

In [25]:
print("Valores de year:", partidas_pro["year"].value_counts().to_dict())
fechas_2026 = partidas_pro.loc[partidas_pro["year"] == 2026, "date"]
print("Fechas de las filas con year = 2026:", fechas_2026.min(), "a", fechas_2026.max())
print("Ligas con year = 2026:", partidas_pro.loc[partidas_pro["year"] == 2026, "league"].unique().tolist())

Valores de year: {2025: 117552, 2026: 2976}
Fechas de las filas con year = 2026: 2025-07-29 15:10:01 a 2025-12-14 11:40:05
Ligas con year = 2026: ['HLL', 'PRM', 'LVP SL', 'AL', 'EBL', 'CD', 'NACL', 'LFL', 'LFL2', 'LTA S', 'LCP', 'LTA N', 'KeSPA Cup']


- **`year` dice 2026 en 2.976 filas, pero sus fechas son de 2025** (julio a diciembre). Probablemente es el año de la temporada a la que pertenece el torneo, no el año en que se jugó. Si filtro por `year` en vez de `date` me saldrían resultados equivocados.

In [26]:
r = partidas_ranked

print("¿blueGoldDiff es siempre -redGoldDiff?", (r["blueGoldDiff"] == -r["redGoldDiff"]).all())
print("¿blueFirstBlood + redFirstBlood siempre suma 1?", ((r["blueFirstBlood"] + r["redFirstBlood"]) == 1).all())
print("¿blueKills es siempre redDeaths?", (r["blueKills"] == r["redDeaths"]).all())
print("¿blueEliteMonsters = blueDragons + blueHeralds?", (r["blueEliteMonsters"] == r["blueDragons"] + r["blueHeralds"]).all())
print("¿blueCSPerMin = blueTotalMinionsKilled / 10?", ((r["blueCSPerMin"] - r["blueTotalMinionsKilled"] / 10).abs() < 1e-9).all())

¿blueGoldDiff es siempre -redGoldDiff? True
¿blueFirstBlood + redFirstBlood siempre suma 1? True
¿blueKills es siempre redDeaths? True
¿blueEliteMonsters = blueDragons + blueHeralds? True
¿blueCSPerMin = blueTotalMinionsKilled / 10? True


En el dataset ranked **muchas columnas son redundantes**: se calculan a partir de otras (`blueGoldDiff` es el negativo de `redGoldDiff`, `blueEliteMonsters` es la suma de dragones y heraldos, `blueCSPerMin` es el CS dividido entre 10). No es un error, pero repetir la misma información puede confundir un análisis o un modelo.

In [27]:
wards = r["blueWardsPlaced"]
q1, q3 = wards.quantile(0.25), wards.quantile(0.75)
limite = q3 + 1.5 * (q3 - q1)

print(wards.describe())
print()
print("Límite para valores atípicos (Q3 + 1,5·RIQ):", limite)
print("Partidas con más wards que ese límite:", (wards > limite).sum())
print("Partidas con más de 100 wards en 10 minutos:", (wards > 100).sum())

count    9879.000000
mean       22.288288
std        18.019177
min         5.000000
25%        14.000000
50%        16.000000
75%        20.000000
max       250.000000
Name: blueWardsPlaced, dtype: float64

Límite para valores atípicos (Q3 + 1,5·RIQ): 29.0
Partidas con más wards que ese límite: 1627
Partidas con más de 100 wards en 10 minutos: 108


- **`blueWardsPlaced` tiene valores atípicos**: la mediana es 16 wards en 10 minutos, pero el máximo es 250. Poner 250 wards en 10 minutos no es posible en una partida normal, así que esos valores parecen errores de registro o partidas fuera de lo común.

### 4.4 Limitaciones de los datos para responder la pregunta

- **El dataset ranked solo llega al minuto 10**, no al 15. Para comparar los dos datasets tengo que usar las columnas `at10` del profesional.
- **La LPL no tiene datos a los 15 minutos**, así que las conclusiones a los 15 minutos no representan a la liga china.
- **El dataset ranked no tiene fecha, parche ni región**, así que no sé de qué versión del juego es (en la EV03 vi que es una instantánea antigua). Los objetivos del juego han cambiado (por ejemplo, las larvas del vacío `void_grubs` solo existen en el profesional de 2025).
- **Los dos datasets miden cosas parecidas con nombres y estructuras distintas**: uno tiene una fila por equipo y otro una fila por partida con columnas azul/rojo. Para compararlos hay que transformarlos a la misma forma.
- **Partidas profesionales y ranked no son lo mismo**: en el profesional los equipos están coordinados; en ranked son 5 jugadores que no se conocen.

### 4.5 Tabla de hallazgos y acciones propuestas

In [28]:
hallazgos = pd.DataFrame([
    ["partidas_pro", "Filas que no son partidas",
     "12 filas por gameid (10 jugadores + 2 equipos)",
     "Contar filas como partidas multiplica todo por 12",
     "Filtrar position == 'team' para el análisis por equipo"],
    ["partidas_pro", "Nulos en variables a los 15 min",
     "1.640 filas de equipo sin golddiffat15, todas con datacompleteness = 'partial'",
     "La LPL queda fuera del análisis a los 15 minutos",
     "Excluir las partidas 'partial' y documentar que la LPL no está representada"],
    ["partidas_pro", "Nulos que no aplican",
     "champion vacío en filas de equipo; towers vacío en filas de jugador",
     "Ninguno si se separan las filas; confunde si no",
     "No imputar; separar filas de equipo y de jugador"],
    ["partidas_pro", "Tipo incorrecto en patch",
     "patch es float64: 15.10 aparece como 15.1",
     "Se confunden parches distintos al agrupar",
     "Convertir patch a texto con dos decimales"],
    ["partidas_pro", "Tipo incorrecto en date",
     "date es str",
     "No se puede ordenar ni filtrar por fecha",
     "Convertir con pd.to_datetime()"],
    ["partidas_pro", "year no coincide con date",
     "2.976 filas con year = 2026 y fechas de 2025",
     "Filtrar por year da resultados equivocados",
     "Usar date para el periodo; tratar year como temporada"],
    ["partidas_pro", "Partidas sin primer objetivo",
     "1 sin firstblood, 7 sin firstdragon, 19 sin firstherald",
     "Pocas, pero pueden sesgar el conteo de 'primer objetivo'",
     "Revisarlas una a una y decidir si se excluyen"],
    ["partidas_ranked", "Valores atípicos en wards",
     "blueWardsPlaced: mediana 16, máximo 250",
     "Distorsionan promedios y modelos",
     "Revisar valores sobre el límite RIQ; recortar o excluir los imposibles"],
    ["partidas_ranked", "Columnas redundantes",
     "blueGoldDiff = -redGoldDiff, EliteMonsters = Dragons + Heralds, CSPerMin = CS/10",
     "Información repetida que infla el modelo",
     "Quedarse con una columna por variable"],
    ["partidas_ranked", "Datos solo hasta el minuto 10 y sin parche",
     "No hay columnas at15 ni patch/date",
     "No responde la pregunta a 15 minutos; no se sabe la versión",
     "Comparar con las columnas at10 del profesional y declarar la limitación"],
    ["partidas_ranked", "Identificador numérico",
     "gameId es int64",
     "describe() le calcula un promedio que no significa nada",
     "Convertir gameId a texto"],
], columns=["dataset", "problema", "evidencia", "posible_impacto", "accion_propuesta"])

hallazgos

,dataset,problema,evidencia,posible_impacto,accion_propuesta
0,partidas_pro,Filas que no son partidas,12 filas por gameid (10 jugadores + 2 equipos),Contar filas como partidas multiplica todo por 12,Filtrar position == 'team' para el análisis por equipo
1,partidas_pro,Nulos en variables a los 15 min,"1.640 filas de equipo sin golddiffat15, todas con datacompleteness = 'partial'",La LPL queda fuera del análisis a los 15 minutos,Excluir las partidas 'partial' y documentar que la LPL no está representada
2,partidas_pro,Nulos que no aplican,champion vacío en filas de equipo; towers vacío en filas de jugador,Ninguno si se separan las filas; confunde si no,No imputar; separar filas de equipo y de jugador
3,partidas_pro,Tipo incorrecto en patch,patch es float64: 15.10 aparece como 15.1,Se confunden parches distintos al agrupar,Convertir patch a texto con dos decimales
4,partidas_pro,Tipo incorrecto en date,date es str,No se puede ordenar ni filtrar por fecha,Convertir con pd.to_datetime()
5,partidas_pro,year no coincide con date,2.976 filas con year = 2026 y fechas de 2025,Filtrar por year da resultados equivocados,Usar date para el periodo; tratar year como temporada
6,partidas_pro,Partidas sin primer objetivo,"1 sin firstblood, 7 sin firstdragon, 19 sin firstherald","Pocas, pero pueden sesgar el conteo de 'primer objetivo'",Revisarlas una a una y decidir si se excluyen
7,partidas_ranked,Valores atípicos en wards,"blueWardsPlaced: mediana 16, máximo 250",Distorsionan promedios y modelos,Revisar valores sobre el límite RIQ; recortar o excluir los imposibles
8,partidas_ranked,Columnas redundantes,"blueGoldDiff = -redGoldDiff, EliteMonsters = Dragons + Heralds, CSPerMin = CS/10",Información repetida que infla el modelo,Quedarse con una columna por variable
9,partidas_ranked,Datos solo hasta el minuto 10 y sin parche,No hay columnas at15 ni patch/date,No responde la pregunta a 15 minutos; no se sabe la versión,Comparar con las columnas at10 del profesional y declarar la limitación


## Conclusiones

1. **El dataset profesional sirve para responder la pregunta, pero no completo.** Tiene las variables a los 15 minutos y el resultado, y en todas las partidas hay un solo ganador. Sin embargo, las 820 partidas de datos parciales (casi todas de la LPL) no tienen esas variables y deben excluirse, y eso hay que aclararlo en las conclusiones.
2. **Hay que entender qué es cada fila antes de calcular.** En el profesional una partida son 12 filas, y muchos nulos solo indican que el dato no aplica a ese tipo de fila. Para el análisis se deben usar las filas de equipo.
3. **El dataset ranked está limpio pero es limitado.** No tiene nulos ni duplicados, pero solo llega al minuto 10, no indica versión del juego, repite información en varias columnas y tiene valores atípicos en los wards.
4. **Antes de analizar hay que preparar los datos:** filtrar filas de equipo, excluir partidas parciales, convertir `patch`, `date` y `gameId` a tipos correctos, revisar los atípicos y llevar los dos datasets a la misma forma para compararlos a los 10 minutos.

Con estas acciones los datos quedan en condiciones de continuar con el proyecto.